# Batch: banded + morphological vs manual (all cortical structures)

The single-structure head-to-head, extended to every cortical (subject, probe, structure)
with manual labels. Both detectors are scored against the manual labels over the common
morphological detection channels with their faithful footprints: morphological at LLAS,
CLAS and BLAS, and two banded band definitions at the operating point (post = 80 ms),
`fixed_tiled` (rollout) and `greedy_fr` (FR-equalized comparison baseline), with params
explicit per pass below.

Engine: `cnpix_local_sleep.evaluation.head_to_head.head_to_head_experiment`. Each
structure is loaded once and both banded passes run on the shared trains; `greedy_fr` is
skipped on low-firing-rate structures, so its column may cover fewer structures than
`fixed_tiled` (see the per-row `count`). This re-detects banded per structure, a
deliberate long run, and the cross-subject numbers are a first look rather than a settled
result.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cnpix_local_sleep.evaluation import head_to_head
from cnpix_local_sleep.unit_based import banded
from cnpix_local_sleep.unit_based import const as ub_const

eval_name = "NREM"
mua_source = "full48h"

# Banded detector passes scored alongside morphological, params explicit per pass. fixed_tiled
# is the rollout operating point (asserted == ROLLOUT_CONFIG); greedy_fr is the FR-equalized
# comparison baseline (skipped per structure if it can't build bands on low FR). Each
# structure is loaded once and shared across both passes.
BANDED_PASSES = {
    "banded-fixed_tiled": {
        "algo": "sticky",  # sticky Poisson-HMM (off_rate_max=0.0 == cap0)
        "band_definition": "fixed_tiled",  # sliding fixed-size depth windows
        "band_sizes": [250.0],  # um; single 250 um scale
        "tile_start": "superficial",
        "param_strategy": "shared",
        "min_band_off_duration": 0.05,  # s; pre-merge floor
        "min_merged_off_duration": 0.08,  # s; post floor (dropped + swept via post_ms)
    },
    "banded-greedy_fr": {
        "algo": "sticky",
        "band_definition": "greedy_fr",  # FR-equalized greedy bands (band_sizes unused)
        "param_strategy": "shared",
        "min_band_off_duration": 0.05,
        "min_merged_off_duration": 0.08,
    },
}
assert BANDED_PASSES["banded-fixed_tiled"] == banded.ROLLOUT_CONFIG, (
    "fixed_tiled pass has diverged from banded.ROLLOUT_CONFIG -- intentional? update this."
)

In [ ]:
# Parameters not set in the configs above but that still govern each pass (engine defaults),
# printed live from the library so this record stays accurate if the defaults change.
from on_off_detection.spatial_off import SPATIAL_PARAMS

MERGE_KEYS = (
    "min_depth_overlap",
    "min_shared_duration_overlap",
    "nearby_off_max_time_diff",
)
for name, cfg in BANDED_PASSES.items():
    print(f"== {name} ==")
    print("  config:", cfg)
    print(
        "  sticky engine (shared across bands):",
        ub_const.UNIT_BASED_PARAMS[cfg["algo"]],
        " # off_rate_max=0.0 == cap0",
    )
    print(
        "  band-inclusion floor: band_min_units=%s, band_min_keep_fr=%s Hz"
        % (SPATIAL_PARAMS["band_min_units"], SPATIAL_PARAMS["band_min_keep_fr"])
    )
    if cfg["band_definition"] == "fixed_tiled":
        step = cfg["band_sizes"][0] * (1 - SPATIAL_PARAMS["band_overlap"])
        print(
            f"  band tiling: band_overlap={SPATIAL_PARAMS['band_overlap']} -> "
            f"{cfg['band_sizes'][0]:.0f} um window every {step:.0f} um"
        )
    else:  # greedy_fr builds bands to hit an FR target (band_sizes unused)
        print(
            "  greedy targets: band_min_fr=%s Hz, band_min_size=%s um, band_fr_overlap=%s"
            % (
                SPATIAL_PARAMS["band_min_fr"],
                SPATIAL_PARAMS["band_min_size"],
                SPATIAL_PARAMS["band_fr_overlap"],
            )
        )
    print(
        "  cross-band merge: "
        + ", ".join(f"{k}={SPATIAL_PARAMS[k]}" for k in MERGE_KEYS)
    )

## Run across all labeled cortical structures

In [ ]:
df = head_to_head.head_to_head_experiment(
    eval_name=eval_name,
    mua_source=mua_source,
    post_ms=(80,),
    scope="detection",
    banded_passes=BANDED_PASSES,
)
print(
    f"{len(df)} rows | "
    f"{df[['subject', 'probe', 'structure']].drop_duplicates().shape[0]} structures | "
    f"by method/label: {df.groupby(['method', 'label']).size().to_dict()}"
)
df.head(10)

In [ ]:
import wisc_ecephys_tools as wet

out = (
    wet.get_sglx_project("offproj").get_experiment_directory(
        "novel_objects_deprivation"
    )
    / f"manual_vs_banded_and_morphological_{mua_source}_NREM.parquet"
)
df.to_parquet(out)
print("wrote", out)

## Summary: mean ± std across structures, per method/label


In [ ]:
# Banded rows keep their pass label (fixed_tiled / greedy_fr, both at post=80); mua keep theirs.
df["row"] = np.where(
    df["method"] == "banded",
    df["label"].astype(str),
    "mua-" + df["label"].astype(str),
)
metric_cols = ["F1", "IoU", "sensitivity", "precision", "n_off"]
summary = df.groupby("row")[metric_cols].agg(["mean", "std", "count"])
summary.round(3)

In [ ]:
print(summary.round(3))

## Mean F1 / IoU by method

In [ ]:
g = df.groupby("row")
order = ["banded-fixed_tiled", "banded-greedy_fr", "mua-llas", "mua-clas", "mua-blas"]
order = [o for o in order if o in g.groups]
means_f1 = [g.get_group(o)["F1"].mean() for o in order]
std_f1 = [g.get_group(o)["F1"].std() for o in order]
means_iou = [g.get_group(o)["IoU"].mean() for o in order]
std_iou = [g.get_group(o)["IoU"].std() for o in order]
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - 0.2, means_f1, 0.4, yerr=std_f1, capsize=4, label="F1", color="#4C72B0")
ax.bar(x + 0.2, means_iou, 0.4, yerr=std_iou, capsize=4, label="IoU", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=20, ha="right")
ax.set_ylim(0, 1)
ax.set_title(
    f"banded vs morphological vs manual: detection scope, mua={mua_source} (mean ± std)"
)
ax.legend()
plt.tight_layout()
plt.show()

## Per-structure F1: greedy_fr vs fixed_tiled

Structures where greedy_fr failed (low FR) are absent from the scatter.

In [ ]:
piv = df.pivot_table(
    index=["subject", "probe", "structure"], columns="row", values="F1", aggfunc="first"
)
if {"banded-fixed_tiled", "banded-greedy_fr"} <= set(piv.columns):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(
        piv["banded-fixed_tiled"],
        piv["banded-greedy_fr"],
        alpha=0.7,
        edgecolors="white",
        s=55,
    )
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("fixed_tiled F1")
    ax.set_ylabel("greedy_fr F1")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(
        "per-structure F1: greedy_fr vs fixed_tiled (above diag = greedy wins)"
    )
    plt.tight_layout()
    plt.show()